# KV-Cache & Paged Attention

A refresher on the **KV cache** — the single most important inference optimization for
autoregressive transformers — and **paged attention**, the memory-management trick (from
vLLM) that makes serving the cache efficient at scale.

**Domain:** LLM Inference, Training & Optimization  ·  **recommended addition**  ·  **runnable:** yes

## 1. What & Why

A transformer generates text one token at a time. To produce token `t`, self-attention needs
the **keys** and **values** of *every* previous token `0..t-1`. The naive approach recomputes
all of those keys and values from scratch on every step — turning generation into an
**O(T²)** firehose of redundant matrix multiplies.

The insight is that for a given token, its key and value vectors **never change** once computed
(attention is causal — past tokens don't see the future). So compute them **once**, stash them
in a per-layer **KV cache**, and on each new step only project the *one* new token's K/V and
append. Generation drops from **O(T²)** to **O(T)** projection work, and decode becomes
memory-bandwidth bound rather than compute bound. Every production inference stack does this;
`use_cache=True` is the default in 🤗 Transformers.

The catch is **memory**. The cache grows linearly with sequence length and can dwarf the model
weights for long contexts — and naively reserving a contiguous, max-length slab per request
wastes most of it. **Paged attention** solves this by storing the cache in fixed-size **blocks**
(like OS virtual-memory pages) allocated on demand, slashing fragmentation and enabling cache
sharing across requests.

**Reach for the KV cache:** always, for any autoregressive decoding. **Reach for paged
attention:** when you *serve* models (many concurrent, variable-length requests) and want high
throughput — it's what lets vLLM pack far more sequences into the same GPU.

## 2. Mental Model

**KV cache = a growing notebook the model writes once and re-reads forever.**

Each token, when first processed, writes its key and value into the notebook. It never edits
those pages again. Every future token just *reads* the whole notebook to decide where to attend.
Without the cache you'd re-derive every earlier page from the raw text on every single step.

**Paged attention = virtual memory for that notebook.** Instead of handing each request one
giant contiguous binder sized for the worst case, the allocator gives out small fixed-size
**blocks** (e.g. 16 tokens each) as the sequence grows, and a per-sequence **block table** maps
logical positions → physical blocks. Blocks need not be contiguous, identical prefixes can
**share** the same physical blocks (copy-on-write), and freeing is just returning blocks to a pool.

```
Two phases of decoding:
  PREFILL  (prompt):  process all N prompt tokens in ONE pass → fill cache with N K/V entries
  DECODE   (per step): project 1 new token → append 1 K/V entry → attend over the whole cache

Contiguous reservation              Paged attention (block size = 4)
  req A: [#### #### .... ....]        pool: [B0][B1][B2][B3][B4][B5] ...
         reserved max, mostly idle    A -> B0,B3   (8 tokens, 2 blocks)
  req B: [## .. .... ....]            B -> B1      (2 tokens, 1 block)
         huge waste                   waste = at most (block-1) tokens / seq
```

## 3. Key Concepts

- **KV cache** — per-layer store of the key and value vectors for all past tokens. Size scales as
  `2 · batch · layers · kv_heads · head_dim · seq_len · dtype_bytes` (the `2` = keys *and* values).
- **Prefill vs decode** — *prefill* processes the whole prompt in one parallel pass (compute-bound);
  *decode* emits one token per pass reusing the cache (memory-bandwidth-bound). They have very
  different performance characteristics, which is why TTFT and inter-token latency are reported
  separately.
- **Why decode is bandwidth-bound** — each step streams all weights *and* the entire KV cache
  through the accelerator to emit one token; the matmuls are tiny, the data movement dominates.
- **Multi-Query / Grouped-Query Attention (MQA/GQA)** — share K/V across attention heads so
  `kv_heads ≪ query_heads`. This shrinks the KV cache directly (e.g. Llama-2-70B uses 8 KV heads
  vs 64 query heads → 8× smaller cache) — the main lever for fitting long contexts.
- **Paged attention** — store the cache in fixed-size **blocks** allocated on demand via a per-sequence
  **block table**, eliminating the need for one contiguous max-length allocation.
- **Internal vs external fragmentation** — contiguous reservation suffers *external* waste (reserved
  but unused tail). Paging trades it for bounded *internal* waste (at most `block_size − 1` tokens
  per sequence, in the last partial block).
- **Prefix sharing / KV reuse** — requests with a common prefix (same system prompt, few-shot
  examples, or a branching beam) can point their block tables at the *same* physical blocks, so the
  shared prefix is cached once. This is *prefix caching* / *automatic prefix caching*.
- **Quantized KV cache** — storing K/V in int8/fp8 roughly halves cache memory at a small quality
  cost; orthogonal to and composable with paging.

## 4. Setup

The worked examples below need only **NumPy** — they reimplement the cache from scratch so the
mechanism is fully visible:

```bash
%pip install numpy
```

The *optional* real-model example in §5.3 also needs Transformers + PyTorch and pulls a small
checkpoint (distilgpt2, ~350 MB):

```bash
%pip install "transformers>=4.40" torch
```

That cell is **gated behind an environment-variable check**, so the notebook executes
top-to-bottom on CPU with no downloads unless you opt in.

In [1]:
# Environment probe — what's available in THIS kernel (no downloads, no GPU needed).
import importlib.util
import sys

import numpy as np


def have(mod: str) -> str:
    return "installed" if importlib.util.find_spec(mod) else "not installed"


print(f"python        : {sys.version.split()[0]}")
print(f"numpy         : {np.__version__}")
for m in ("torch", "transformers"):
    print(f"{m:<14}: {have(m)}")

print("\nWorked examples 1 & 2 are pure-NumPy and run regardless of the above.")

python        : 3.13.7
numpy         : 2.5.0
torch         : installed
transformers  : installed

Worked examples 1 & 2 are pure-NumPy and run regardless of the above.


## 5. Worked Examples

### 5.1 The KV cache is *exact* — same output, far less work

We'll decode a short sequence two ways through a tiny single-head causal self-attention: (a) the
**naive** way that recomputes every token's K/V from scratch each step, and (b) with a **KV cache**
that projects only the new token and appends. The outputs must be **bit-for-bit identical** — the
cache changes *how much you compute*, never *what you compute* — while the projection work collapses
from O(T²) to O(T).

In [2]:
rng = np.random.default_rng(0)
d = 16   # model / head dimension
T = 8    # tokens we decode one at a time

# Fixed projection matrices (one head) and token embeddings revealed left-to-right.
Wq = rng.standard_normal((d, d)) / np.sqrt(d)
Wk = rng.standard_normal((d, d)) / np.sqrt(d)
Wv = rng.standard_normal((d, d)) / np.sqrt(d)
X = rng.standard_normal((T, d))


def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)


def attend(q, K, V):
    """One query attends over cached K, V. Causality is implicit: K/V hold only past+current."""
    return softmax((K @ q) / np.sqrt(d)) @ V


# (a) NAIVE: re-project K and V for the entire prefix on every step.
naive_out, naive_projections = [], 0
for t in range(T):
    ctx = X[: t + 1]                 # everything seen so far
    K, V = ctx @ Wk, ctx @ Wv        # recomputed from scratch each step
    naive_projections += ctx.shape[0]
    naive_out.append(attend(X[t] @ Wq, K, V))

# (b) CACHED: project only the new token's K/V and append to the cache.
cached_out, cached_projections = [], 0
K_cache = np.empty((0, d))
V_cache = np.empty((0, d))
for t in range(T):
    xt = X[t]
    K_cache = np.vstack([K_cache, xt @ Wk])   # one new row
    V_cache = np.vstack([V_cache, xt @ Wv])
    cached_projections += 1
    cached_out.append(attend(xt @ Wq, K_cache, V_cache))

naive_out, cached_out = np.array(naive_out), np.array(cached_out)
print("outputs identical      :", np.allclose(naive_out, cached_out))
print(f"K/V row-projections    : naive={naive_projections}  cached={cached_projections}")
print(f"naive grows as T(T+1)/2 = {T * (T + 1) // 2}  (O(T²));  cached grows as T = {T}  (O(T))")

outputs identical      : True
K/V row-projections    : naive=36  cached=8
naive grows as T(T+1)/2 = 36  (O(T²));  cached grows as T = 8  (O(T))


Identical outputs, but the naive loop did **36** token-projections to the cache's **8** — and that
gap widens quadratically with context length. For a 4k-token generation the cache saves ~2000× the
redundant K/V projection work. This is why caching is non-negotiable, not an optimization you toggle.

### 5.2 The cost of the cache: memory growth and why paging wins

The cache isn't free — it grows linearly with sequence length and, for long contexts, can exceed the
model weights. First size a real cache; then see how **contiguous max-length reservation** wastes
memory that **paged** allocation reclaims.

In [3]:
def kv_cache_bytes(seq_len, n_layers, n_kv_heads, head_dim, dtype_bytes=2, batch=1):
    """Bytes for the KV cache. The leading 2 counts BOTH keys and values."""
    return 2 * batch * n_layers * n_kv_heads * head_dim * seq_len * dtype_bytes


# Llama-2-7B in fp16: 32 layers, 32 KV heads (no GQA at 7B), head_dim 128.
cfg = dict(n_layers=32, n_kv_heads=32, head_dim=128, dtype_bytes=2)
print("Llama-2-7B (fp16) KV cache, per sequence:")
for seq in (512, 2048, 8192, 32768):
    print(f"  seq_len={seq:>6}: {kv_cache_bytes(seq, **cfg) / 1e9:6.2f} GB")

# GQA shrinks it: Llama-2-70B uses 8 KV heads vs 64 query heads at the SAME 8k context.
mha = kv_cache_bytes(8192, n_layers=80, n_kv_heads=64, head_dim=128)
gqa = kv_cache_bytes(8192, n_layers=80, n_kv_heads=8, head_dim=128)
print(f"\n70B @ 8k context  MHA(64 kv heads)={mha / 1e9:.1f} GB  vs  GQA(8 kv heads)={gqa / 1e9:.1f} GB"
      f"  → {mha / gqa:.0f}x smaller")

Llama-2-7B (fp16) KV cache, per sequence:
  seq_len=   512:   0.27 GB
  seq_len=  2048:   1.07 GB
  seq_len=  8192:   4.29 GB
  seq_len= 32768:  17.18 GB

70B @ 8k context  MHA(64 kv heads)=21.5 GB  vs  GQA(8 kv heads)=2.7 GB  → 8x smaller


In [4]:
# Paging vs contiguous reservation: 20 concurrent requests of varied length.
rng2 = np.random.default_rng(1)
max_len = 2048   # worst-case context the server must be able to serve
block = 16       # tokens per KV block (vLLM-style)
seq_lens = rng2.integers(50, 600, size=20)

used = int(seq_lens.sum())                                       # tokens actually live
contiguous = max_len * len(seq_lens)                             # reserve max for everyone
paged = sum(int(np.ceil(L / block)) * block for L in seq_lens)   # whole blocks on demand

print(f"live tokens               : {used}")
print(f"contiguous (reserve max)  : {contiguous:>6}  → {100 * (1 - used / contiguous):4.1f}% wasted")
print(f"paged (block={block})            : {paged:>6}  → {100 * (1 - used / paged):4.1f}% wasted")
print(f"\npaging fits the same load in {contiguous / paged:.1f}x less KV memory here.")
print("Paged waste is bounded: at most (block-1) tokens per sequence sit in a partial last block.")

live tokens               : 6255
contiguous (reserve max)  :  40960  → 84.7% wasted
paged (block=16)            :   6416  →  2.5% wasted

paging fits the same load in 6.4x less KV memory here.
Paged waste is bounded: at most (block-1) tokens per sequence sit in a partial last block.


Contiguous reservation throws away the overwhelming majority of reserved memory because every
request must be sized for the worst case it might reach. Paging reserves only whole blocks as the
sequence actually grows, so the only waste is the partial final block — capped at `block_size − 1`
tokens per sequence. That reclaimed memory is exactly what lets a serving engine hold **many more**
concurrent sequences on one GPU, which is where throughput comes from.

### 5.3 The real cache in 🤗 Transformers (optional, gated)

In practice the cache is automatic: `model(..., use_cache=True)` returns `past_key_values`, which you
feed back on the next step. The cell downloads a small checkpoint, so it's gated behind an env var
and skipped by default. Set `RUN_HF_KVCACHE=1` to actually run it.

In [5]:
import os

if os.getenv("RUN_HF_KVCACHE") and importlib.util.find_spec("transformers"):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained("distilgpt2")
    model = AutoModelForCausalLM.from_pretrained("distilgpt2")
    inputs = tok("The KV cache stores", return_tensors="pt")

    with torch.no_grad():
        out = model(**inputs, use_cache=True)

    pkv = out.past_key_values
    try:                                   # newer transformers: a Cache object
        k0, n = pkv.key_cache[0], len(pkv.key_cache)
    except AttributeError:                 # legacy: tuple-of-tuples
        k0, n = pkv[0][0], len(pkv)
    print("layers cached  :", n)
    print("key shape      :", tuple(k0.shape), "= (batch, kv_heads, seq_len, head_dim)")
    print("On the next step you pass past_key_values back so only the new token is projected.")
else:
    print("Skipped (set RUN_HF_KVCACHE=1 and install transformers+torch to run).")
    print("Call shape:")
    print("  out = model(**inputs, use_cache=True)")
    print("  past = out.past_key_values            # K/V for every layer")
    print("  out = model(next_tok, past_key_values=past, use_cache=True)  # reuse, project 1 token")
    print("\nKey point: use_cache=True makes generate() O(T) instead of O(T²). It is the default.")

Skipped (set RUN_HF_KVCACHE=1 and install transformers+torch to run).
Call shape:
  out = model(**inputs, use_cache=True)
  past = out.past_key_values            # K/V for every layer
  out = model(next_tok, past_key_values=past, use_cache=True)  # reuse, project 1 token

Key point: use_cache=True makes generate() O(T) instead of O(T²). It is the default.


## 6. Gotchas & Pitfalls

- **The cache is your real context-length limit.** You usually run out of KV memory long before you
  run out of model capacity. Long-context serving is a *memory* problem first — budget cache bytes,
  not just parameters.
- **`use_cache` is incompatible with gradient checkpointing.** During training/fine-tuning HF will
  warn and silently disable the cache. That's correct — caching is a *decode-time* optimization, not
  a training one. Don't fight the warning.
- **Don't recompute when you meant to cache.** Re-running `model(full_sequence)` each step (instead of
  passing `past_key_values`) is the classic accidental O(T²) loop — correct output, brutal latency.
- **Position IDs must continue, not reset.** When you feed only the new token with a cache, its
  position id is the running offset, not 0. Frameworks handle this; hand-rolled loops often get it
  wrong and produce garbage after the first step.
- **Speculative decoding needs cache rollback.** On a rejected draft token you must truncate the
  target's KV cache back to the last accepted position; a stale cache corrupts everything after.
- **Sliding-window / StreamingLLM ≠ free infinite context.** Evicting old KV entries caps memory but
  *loses* the information in evicted tokens — it's lossy, unlike the exact full cache.
- **Paging has overhead.** The block-table indirection and non-contiguous gathers cost a little per
  step; the win is throughput at scale, not single-stream latency. For one short request a contiguous
  cache can be marginally faster.
- **Quantizing the KV cache is not free.** int8/fp8 K/V halves memory but can dent quality on
  long-context or precision-sensitive tasks. Measure before shipping it.

## 7. When to Use vs Alternatives

| Technique | What it does | Cost / trade-off | Best when |
|---|---|---|---|
| **KV cache** | Reuse past K/V instead of recomputing | Linear memory in seq_len | **Always**, for any autoregressive decode |
| **Paged attention** | Block-based, on-demand cache allocation | Indirection overhead | Serving many concurrent, variable-length requests |
| **MQA / GQA** | Fewer KV heads shared across query heads | Slight quality change (train-time choice) | You need long contexts or large batches to fit |
| **Prefix / automatic prefix caching** | Share KV blocks for common prefixes | Bookkeeping, copy-on-write | Shared system prompts, few-shot, beam/branching |
| **KV quantization (int8/fp8)** | Store K/V in lower precision | Small accuracy loss | Memory-bound and quality budget allows |
| **Sliding window / StreamingLLM** | Evict old KV entries, cap cache size | **Lossy** — drops old context | Effectively-infinite streams where old tokens don't matter |
| **No cache (recompute)** | Recompute K/V every step | O(T²) compute | Essentially never — only a single forward / prefill |

**Rule of thumb:** the KV cache itself is not optional — turn it on and never think about it again.
The *interesting* decisions are about its **memory**: reach for **paged attention + prefix caching**
when serving, choose **GQA** models for long contexts, and add **KV quantization** only when you've
measured a memory wall and can afford a little quality. These compose — vLLM stacks paging, prefix
caching, and (optionally) FP8 KV on top of GQA models.

See also: [`vllm`](vllm.ipynb) (paged attention in production), [`flash-attention`](flash-attention.ipynb)
(the attention kernel the cache feeds), [`speculative-decoding`](speculative-decoding.ipynb) (needs
cache rollback), and [`quantization-gptq-awq`](quantization-gptq-awq.ipynb) (weight quantization,
orthogonal to KV quantization).

## 8. Resources

- **Kwon et al. (2023), *Efficient Memory Management for LLM Serving with PagedAttention*** — the
  vLLM paper that introduced paged attention: https://arxiv.org/abs/2309.06180
- **vLLM docs — *Automatic Prefix Caching*** (how KV blocks are shared across requests):
  https://docs.vllm.ai/en/latest/features/automatic_prefix_caching.html
- **Shazeer (2019), *Fast Transformer Decoding: One Write-Head is All You Need*** — the original
  Multi-Query Attention paper: https://arxiv.org/abs/1911.02150
- **Ainslie et al. (2023), *GQA: Training Generalized Multi-Query Transformer Models*:**
  https://arxiv.org/abs/2305.13245
- **Hugging Face docs — *Best practices for generation with caching* / KV cache strategies:**
  https://huggingface.co/docs/transformers/main/en/kv_cache
- **Xiao et al. (2023), *Efficient Streaming Language Models with Attention Sinks* (StreamingLLM):**
  https://arxiv.org/abs/2309.17453